# Tenant Satisfaction Measures: Step 2, join and explore the operational data

In step 1 I only looked at the tenant survey. But the TSM file has a second sheet, the **management information**, which is about what landlords actually did: repairs completed on time, complaints received, anti-social behaviour (ASB) cases and so on. I wanted to put the two together, because "what landlords do" against "how tenants feel" is the interesting bit.

My plan for this notebook:
1. Load both sheets and join them on landlord code
2. Look at the management information and what to watch out for
3. Compare the operational measures across landlord sizes
4. See which operational measures move with satisfaction

I made it run on its own, so step 1 doesn't need to be open.

## 1. Load both sheets

In [1]:
import pandas as pd

# Same upload trick as step 1: a button in Colab, or the data folder on my own computer.
try:
    from google.colab import files
    uploaded = files.upload()
    path = list(uploaded.keys())[0]
except ImportError:
    path = "../data/2025_TSM_Full_Data_v1.1_FINAL.xlsx"

In [2]:
# The management information sheet has one more code than step 1: [U] means a figure
# couldn't be calculated because it would mean dividing by zero (e.g. a landlord with no lifts).
missing_codes = ["[X]", "[Z]", "[E]", "[U]"]

# The two sheets have different numbers of note rows at the top, so different header rows.
lcra = pd.read_excel(path, sheet_name="TSM25_LCRA_Perception", header=10, na_values=missing_codes)
mi = pd.read_excel(path, sheet_name="TSM25_Management_Info", header=9, na_values=missing_codes)
print("Perception:", lcra.shape, " Management info:", mi.shape)

Perception: (353, 135)  Management info: (359, 104)


## 2. Tidy the perception data

This is the same tidying as step 1, just squashed into fewer lines.

In [3]:
tp_pattern = r"^Proportion of respondents who report that they are satisfied.*\((TP\d{2})\)\d*$"
# A dictionary comprehension: the same loop as step 1 on one line.
# The := ("walrus") operator lets me save the match and test it in the same go.
rename_map = {c: m for c in lcra.columns
              if pd.notna(m := pd.Series([c]).str.extract(tp_pattern).iloc[0, 0])}
rename_map.update({
    "Landlord name": "landlord", "Landlord code": "code", "Landlord type": "type",
    "Landlord group size (homes owned, LCRA and LCHO combined)": "size_band",
    "Landlord predominant region": "region",
    "Predominant collection method": "method",
})
tp_cols = sorted(v for v in rename_map.values() if v.startswith("TP"))
perception = lcra.rename(columns=rename_map)[
    ["landlord", "code", "type", "size_band", "region", "method"] + tp_cols]
perception.shape

(353, 18)

## 3. Pick out the management information measures

Every measure's column name includes its official TSM code, for example `(RP02 1)` for non-emergency repairs completed on time. I decided to find columns by that code rather than by their position, because if RSH moves the columns around next year, the code will still work.

One thing that caught me out: the complaints measures appear three times (for rented homes, shared ownership, and both combined). I took the **LCRA** (low cost rental) versions, because those match the tenants who answered the survey.

In [4]:
# A small function to find exactly one column containing all the bits of text I give it.
# The assert makes it stop with an error if it finds none or more than one,
# so I can't accidentally pick up the wrong column without noticing.
def find_col(df, must_contain, starts_with=None):
    """Return the one column whose name contains every text in must_contain."""
    hits = [c for c in df.columns
            if all(t in c for t in must_contain)
            and (starts_with is None or c.startswith(starts_with))]
    assert len(hits) == 1, f"Expected 1 match for {must_contain}, found {len(hits)}: {hits}"
    return hits[0]

# The measures I want, as {my short name: (text the column must contain, what it starts with)}.
mi_measures = {
    # short name:            (text the column must contain,              starts with)
    "gas_safety_pc":          (["(BS01)"],                                 "Proportion"),
    "asb_per_1000":           (["(NM01 1)"],                               "Number of anti-social"),
    "non_decent_pc":          (["(RP01)"],                                 "Proportion"),
    "repairs_on_time_pc":     (["(RP02 1)"],                               "Proportion"),
    "emergency_on_time_pc":   (["(RP02 2)"],                               "Proportion"),
    "repair_target_days":     (["target timescale for non-emergency repairs used to calculate"], "What is your"),
    "complaints_s1_per_1000": (["(CH01 1)"],                               "LCRA"),
    "complaints_s1_on_time_pc": (["(CH02 1)"],                             "LCRA"),
    "complaints_s2_per_1000": (["(CH01 2)"],                               "LCRA"),
    "complaints_s2_on_time_pc": (["(CH02 2)"],                             "LCRA"),
}

# Build a new table with the landlord code and one column per measure.
mi_cols = list(mi_measures)
management = pd.DataFrame({"code": mi["Landlord code"]})
for short, (texts, start) in mi_measures.items():
    col = find_col(mi, texts, start)
    # errors="coerce" turns anything that still isn't a number into a blank (NaN).
    management[short] = pd.to_numeric(mi[col], errors="coerce")

management.head()

,code,gas_safety_pc,asb_per_1000,non_decent_pc,repairs_on_time_pc,emergency_on_time_pc,repair_target_days,complaints_s1_per_1000,complaints_s1_on_time_pc,complaints_s2_per_1000,complaints_s2_on_time_pc
0,L4240,96.9,37.5,0.2,73.9,85.3,20.0,199.9,70.2,47.1,73.7
1,L4172,99.9,55.6,0.0,94.7,98.7,64.0,69.9,79.4,20.4,75.0
2,L4511,100.0,28.0,0.0,81.3,93.3,28.0,72.2,52.9,13.4,38.5
3,L4229,100.0,11.5,0.0,99.9,99.9,28.0,35.7,100.0,7.8,100.0
4,45UB,99.8,18.3,0.8,NaN,60.7,90.0,68.0,24.6,12.3,25.8


## 4. Join the two tables

I joined on **landlord code** rather than name, because codes are unique and names can be written slightly differently in different places.

An *inner* join only keeps landlords that are in both tables, so before joining I checked who would drop out.

In [5]:
# Sets make it easy to see which codes are in one table but not the other.
in_perception = set(perception["code"])
in_management = set(management["code"])

print("In perception only: ", sorted(in_perception - in_management))
print("In management only: ", len(in_management - in_perception), "landlords")

# validate="one_to_one" is a safety check: pandas complains if a code appears twice in either table.
df = perception.merge(management, on="code", how="inner", validate="one_to_one")
print("Joined:", df.shape)

In perception only:  ['43UK']
In management only:  7 landlords
Joined: (352, 28)


Only one landlord drops out of the survey side: **Tandridge District Council**. I looked it up in the `TSM25_Man_Info_Not_Inc` sheet, and RSH excluded its management information because of data quality. The 7 landlords that only appear in the management information don't have a general needs tenant survey in this release (for example, they only have shared owners).

## 5. Profile the management information

In [6]:
# The same describe() summary as step 1, for the operational measures.
df[mi_cols].describe().T.round(1)

,count,mean,std,min,25%,50%,75%,max
gas_safety_pc,352.0,99.7,1.2,83.1,99.8,100.0,100.0,100.0
asb_per_1000,352.0,46.8,44.9,1.8,22.7,36.2,57.6,559.0
non_decent_pc,351.0,3.1,7.0,0.0,0.0,0.5,3.2,71.8
repairs_on_time_pc,350.0,80.8,12.1,38.1,74.9,82.5,89.1,100.0
emergency_on_time_pc,351.0,92.4,9.0,45.1,89.1,94.9,98.9,100.0
repair_target_days,352.0,41.1,35.1,5.0,20.0,28.0,60.0,260.0
complaints_s1_per_1000,352.0,58.7,34.7,5.3,34.4,53.9,72.3,199.9
complaints_s1_on_time_pc,351.0,82.4,18.7,14.5,72.8,89.9,96.8,100.0
complaints_s2_per_1000,352.0,10.8,8.5,0.5,5.2,8.4,13.6,47.1
complaints_s2_on_time_pc,352.0,81.5,21.4,8.6,70.3,88.9,100.0,100.0


**Things I need to keep in mind before comparing anyone:**

- **Gas safety (BS01)** is at or near 100% for almost every landlord. If nearly everyone scores the same, it can't explain why satisfaction differs, so I've left the safety measures out.
- **Repairs "on time"** is measured against **each landlord's own target**, and the targets are all over the place (see the next cell). 90% on time against a 20-day target isn't the same as 90% against a 90-day target.
- **Non-decent homes** is self-reported, and depends on how recently each landlord surveyed its homes (RSH points this out in the sheet).
- **ASB cases per 1,000 homes** probably says as much about how a landlord records ASB as how much ASB there actually is.

In [7]:
# How different are the repair targets? describe() for the spread, value_counts() for the most common ones.
print(df["repair_target_days"].describe().round(1))
df["repair_target_days"].value_counts().head(8)

count    352.0
mean      41.1
std       35.1
min        5.0
25%       20.0
50%       28.0
75%       60.0
max      260.0
Name: repair_target_days, dtype: float64


repair_target_days
20.0    100
28.0     59
60.0     36
90.0     22
40.0     17
21.0     15
30.0     15
64.0     12
Name: count, dtype: int64

## 6. Do the operational measures differ by landlord size?

Before linking anything to satisfaction, I wanted a feel for whether bigger and smaller landlords look different on the operational side. I used medians rather than averages because several of these measures have extreme outliers.

In [8]:
# Every operational measure except gas safety (which is near 100% for everyone).
ops = [c for c in mi_cols if c != "gas_safety_pc"]
# The median of each measure for each size band. .T flips it so the measures run down the side.
by_size = df.groupby("size_band")[ops].median().T
# Put the size bands in a sensible order, smallest first.
by_size[["<5,000", "5,000-40,000", ">40,000"]].round(1)

size_band,"<5,000","5,000-40,000",">40,000"
asb_per_1000,29.8,40.8,48.9
non_decent_pc,0.6,0.5,0.4
repairs_on_time_pc,85.9,81.4,73.5
emergency_on_time_pc,93.9,96.0,89.4
repair_target_days,25.0,28.0,40.0
complaints_s1_per_1000,49.2,52.9,72.9
complaints_s1_on_time_pc,91.5,89.3,79.4
complaints_s2_per_1000,7.6,8.4,17.9
complaints_s2_on_time_pc,95.0,86.8,70.9


**The biggest landlords look weaker on most operational measures.** Those with over 40,000 homes have a median of 73.5% of repairs on time, against 85.9% for landlords with under 5,000. They have more than twice as many stage 2 complaints per 1,000 homes (17.9 against 7.6), answer fewer complaints on time, and set longer repair targets (40 days against 25).

There are only 19 landlords in the biggest band though, so I'm treating this as a pattern to keep in mind rather than a finding. It's also a good reason to allow for size when I build the model in step 3.

## 7. Which operational measures move with satisfaction?

I used **Spearman correlation** here rather than the usual (Pearson) one. Spearman compares rankings instead of the raw numbers, which I read is better when the data has extreme outliers, and this data definitely does: ASB ranges from under 1 to over 500 cases per 1,000 homes. A couple of extreme landlords could throw off a normal correlation.

How I read the numbers: +1 means landlords that rank high on one measure always rank high on the other, −1 means always the opposite, and 0 means no relationship.

I only used housing associations, because councils are different in ways the data doesn't capture (older stock, different funding, different areas).

In [9]:
# All the operational measures except gas safety, against four of the survey measures.
analysis_cols = [c for c in mi_cols if c != "gas_safety_pc"]
outcomes = ["TP01", "TP02", "TP03", "TP09"]   # overall, repairs, repair time, complaints handling

# "has" = housing associations. .loc keeps the operational measures as rows and survey measures as columns.
has = df[df["type"] == "PRP"]
corr = has[analysis_cols + outcomes].corr(method="spearman").loc[analysis_cols, outcomes]
print(f"Spearman correlations, {len(has)} housing associations")
corr.round(2).sort_values("TP01")

Spearman correlations, 195 housing associations


,TP01,TP02,TP03,TP09
complaints_s2_per_1000,-0.65,-0.63,-0.59,-0.54
complaints_s1_per_1000,-0.46,-0.45,-0.46,-0.32
non_decent_pc,-0.39,-0.36,-0.36,-0.29
repair_target_days,-0.03,0.01,-0.03,-0.10
asb_per_1000,0.00,0.01,-0.03,0.00
emergency_on_time_pc,0.29,0.28,0.31,0.26
repairs_on_time_pc,0.31,0.30,0.39,0.31
complaints_s1_on_time_pc,0.31,0.34,0.35,0.26
complaints_s2_on_time_pc,0.35,0.34,0.35,0.35


**What I take from this:**

- **Stage 2 complaints per 1,000 homes** has by far the strongest link with low satisfaction (−0.65 with overall satisfaction). Stage 2 is when a tenant wasn't happy with the first response and escalated their complaint.
- **Non-decent homes** and **stage 1 complaints** also go with lower satisfaction.
- **Repairs completed on time** goes with higher satisfaction, and most strongly with **TP03** (satisfaction with repair time, +0.39), which makes sense.
- **Answering complaints on time** also goes with higher satisfaction.
- **The repair target itself** has almost no link. A longer target doesn't seem to make tenants happier or unhappier on its own.
- **ASB cases** has no link at all, which fits with my worry that it reflects recording more than actual ASB.

I have to remember that **correlation isn't causation**. It could be that unhappy tenants complain more, rather than complaints making tenants unhappy. In step 3 I'll test these together, while allowing for survey method and landlord size.

## 8. The survey measures overlap a lot

One more check before I build a model: how closely do the survey measures move together?

In [10]:
# Spearman correlations between five of the survey measures.
has[tp_cols].corr(method="spearman").loc[["TP01", "TP02", "TP03", "TP04", "TP06"],
                                         ["TP01", "TP02", "TP03", "TP04", "TP06"]].round(2)

,TP01,TP02,TP03,TP04,TP06
TP01,1.00,0.88,0.86,0.95,0.93
TP02,0.88,1.00,0.94,0.88,0.82
TP03,0.86,0.94,1.00,0.84,0.80
TP04,0.95,0.88,0.84,1.00,0.91
TP06,0.93,0.82,0.80,0.91,1.00


Most of these are around 0.9, so the survey measures pretty much move together: a tenant who's unhappy overall tends to score everything lower. That means using, say, TP02 to "explain" TP01 wouldn't tell me much. So in step 3 I'm going to explain overall satisfaction (TP01) using **only the management information**, which is also the part a landlord can actually act on.

## What I found in step 2

- **352 landlords** joined up. Only Tandridge District Council dropped out, because RSH excluded its management information.
- The main caveats: **repair targets vary from 5 to 260 days** (the most common are 20 and 28 days), non-decent homes depends on how recently stock was surveyed, and ASB reflects recording practice.
- **The largest landlords look weaker on repairs and complaints**, though there are only 19 of them.
- Across housing associations, **stage 2 complaints** has the strongest link with low satisfaction (−0.65), and **repairs on time** goes with higher satisfaction with repair time (+0.39).
- The survey measures overlap heavily (correlations around 0.9), so I'll explain overall satisfaction using the operational measures only.

**Next:** step 3, a regression model of overall satisfaction.